# YoloV8 Detection
Este Notebook es parte de un proyecto que se puede encontrar [aqui](https://github.com/nel-eleven11/Proyecto2_DataScience), donde se busca diseñar una aplicación de datos para comparar e interactuar con diferentes modelos de visión por computadora. Primero, vamos a empezar instalando las librerías requeridas que incluyen Ultralytics y los modelos Yolo. Adicionalmente, estaremos utilizando Polars en lugar de Pandas por conflictos de dependencias dentro de Kaggle.

In [1]:
# Minimal, let ultralytics bring its own friends
!pip install --upgrade --no-cache-dir "ultralytics[export]" "opencv-python-headless" "polars"

from ultralytics import YOLO
import os
import numpy as np
import polars as pl
from pathlib import Path
import shutil

INFO: pip is looking at multiple versions of opencv-python-headless to determine which version is compatible with other requirements. This could take a while.


Según el output, Ultralytics se instaló de manera correcta. Sin embargo, tenemos un warning por parte de Pip sobre algunas dependencias que podemos ignorar. También podemos realizar un sanity-check simple para verificar que todo esté funcionando

In [2]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.info()

YOLOv8n summary: 129 layers, 3,157,200 parameters, 0 gradients, 8.9 GFLOPs


(129, 3157200, 0, 8.8575488)

## Pre-Procesamiento
A pesar de ya haber realizado un EDA, todavía debemos de preparar los datos en un formato soportado por YoloV8. Primero, vamos a empezar cargando los datos de nuestro dataset.

### Carga de Datos
Al trabajar dentro de Kaggle, podemos importar los datos y los outputs del Notebook de limpieza. Podemos revisar los directorios rápidamente

In [3]:
import os
print(os.listdir("/kaggle/input"))

['00-eda-and-cleaning', 'mosquito-data']


Luego, podemos setear algunas variables que nos serán de utilidad para saber dónde se encuentra la información.

In [4]:
RAW_DATA_PATH = "/kaggle/input/mosquito-data"
EDA_OUTPUT_PATH = "/kaggle/input/00-eda-and-cleaning"

print("raw:", os.listdir(RAW_DATA_PATH))
print("eda:", os.listdir(EDA_OUTPUT_PATH))

raw: ['train_images', 'sample_submission_phase1 (1).csv', 'test_images_phase1', 'test_phase1.csv', 'train.csv']
eda: ['__results__.html', 'val.csv', '__notebook__.ipynb', '__results___files', '__output__.json', 'train.csv', 'test.csv', 'custom.css']


Podemos ver por los resultados, que tenemos cargados ya los resultados de la limpieza en EDA_OUTPUT_PATH/train.csv, test.csv y val.csv respectivamente. Adicionalmente, las imágenes que utilizaremos se encuentran en RAW_DATA_PATH/train_images. Podemos cargar los datos hacia DataFrames utilizando Polars.

In [5]:
train_df = pl.read_csv(os.path.join(EDA_OUTPUT_PATH, "train.csv"))
val_df   = pl.read_csv(os.path.join(EDA_OUTPUT_PATH, "val.csv"))
test_df  = pl.read_csv(os.path.join(EDA_OUTPUT_PATH, "test.csv"))

print("train shape:", train_df.shape)
print("val shape  :", val_df.shape)
print("test shape :", test_df.shape)

print("Columns:", train_df.columns)
print("Class labels:", train_df.select("class_label").unique())

train shape: (6396, 8)
val shape  : (800, 8)
test shape : (800, 8)
Columns: ['img_fName', 'img_w', 'img_h', 'bbx_xtl', 'bbx_ytl', 'bbx_xbr', 'bbx_ybr', 'class_label']
Class labels: shape: (6, 1)
┌────────────────────┐
│ class_label        │
│ ---                │
│ str                │
╞════════════════════╡
│ culex              │
│ aegypti            │
│ culiseta           │
│ albopictus         │
│ japonicus/koreicus │
│ anopheles          │
└────────────────────┘


Luego del sanity check, podemos confirmar que los datos fueron cargados exitosamente. Ahora, Yolo espera que las clases sean mappeadas de manera numérica.

In [6]:
classes = sorted(train_df.select("class_label").unique()["class_label"].to_list())
print("classes:", classes)

class_to_id = {cls: i for i, cls in enumerate(classes)}
print("class_to_id:", class_to_id)

train_df = train_df.with_columns(
    pl.col("class_label")
      .replace(class_to_id)      # <- use dict mapping instead of map_elements
      .cast(pl.Int64)
      .alias("class_id")
)

val_df = val_df.with_columns(
    pl.col("class_label")
      .replace(class_to_id)
      .cast(pl.Int64)
      .alias("class_id")
)

test_df = test_df.with_columns(
    pl.col("class_label")
      .replace(class_to_id)
      .cast(pl.Int64)
      .alias("class_id")
)

train_df.head()

classes: ['aegypti', 'albopictus', 'anopheles', 'culex', 'culiseta', 'japonicus/koreicus']
class_to_id: {'aegypti': 0, 'albopictus': 1, 'anopheles': 2, 'culex': 3, 'culiseta': 4, 'japonicus/koreicus': 5}


img_fName,img_w,img_h,bbx_xtl,bbx_ytl,bbx_xbr,bbx_ybr,class_label,class_id
str,i64,i64,i64,i64,i64,i64,str,i64
"""92715872-3287-4bff-aa61-704797…",2448,3264,1301,1546,1641,2096,"""albopictus""",1
"""82df4b68-0f45-4afe-9215-48488b…",768,1024,220,58,659,808,"""albopictus""",1
"""331ad30a-7564-4478-b863-7bc760…",3456,4608,1169,2364,1586,2826,"""albopictus""",1
"""46f34803-f754-457d-bdb7-e581d3…",1152,2560,198,798,954,1351,"""albopictus""",1
"""5792dd8b-e690-4c3a-b061-a375ab…",3072,4080,1104,1030,2458,2911,"""anopheles""",2


Ahora podemos empezar a construir los directorios para el  modelo

### Construcción Formato YOLO
Yolo tiene un formato específico que se debe seguir para entrenar sus  modelos, por lo cual debemos crear nuevos directorios y re-organizar nuestros datos. Empezando por crear nuestros paths, al igual que algunas otras variables de utilidad para evitar ser desordenados

In [7]:
ROOT = Path("/kaggle/working/mosquito_yolo")

IMG_TRAIN_DIR = ROOT / "images" / "train"
IMG_VAL_DIR   = ROOT / "images" / "val"
LBL_TRAIN_DIR = ROOT / "labels" / "train"
LBL_VAL_DIR   = ROOT / "labels" / "val"

for d in [IMG_TRAIN_DIR, IMG_VAL_DIR, LBL_TRAIN_DIR, LBL_VAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

IMAGE_SRC_DIR = Path(RAW_DATA_PATH) / "train_images"

Podemos correr unos sanity checks sobre los conjuntos de prueba y validación

In [8]:
train_imgs = train_df.select("img_fName").unique()["img_fName"].to_list()
len(train_imgs)

6396

In [9]:
val_imgs = val_df.select("img_fName").unique()["img_fName"].to_list()
len(val_imgs)

800

Ahora, podemos definir una función de utilidad para llenar los directorios dónde tenemos los datos en el formato que espera YOLO

In [10]:
from pathlib import Path
import polars as pl
import shutil

def build_yolo_split(df: pl.DataFrame, src_img_dir: Path, dst_img_dir: Path, dst_lbl_dir: Path):
    """
    Crea imágenes y labels YOLO para un split (train/val/test)
    a partir de un DataFrame Polars con columnas:
    img_fName, img_w, img_h, bbx_xtl, bbx_ytl, bbx_xbr, bbx_ybr, class_id
    """
    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    # lista de imágenes únicas en el split
    img_names = df.select("img_fName").unique()["img_fName"].to_list()

    for img_name in img_names:
        # Filtrar todas las filas de esa imagen
        group = df.filter(pl.col("img_fName") == img_name)

        src_img = src_img_dir / img_name
        if not src_img.exists():
            print("[WARN] no existe imagen:", src_img)
            continue

        # Copiar imagen
        dst_img = dst_img_dir / img_name
        if not dst_img.exists():
            shutil.copy2(src_img, dst_img)

        # Obtener ancho/alto (primer row)
        w = float(group.select("img_w")[0, 0])
        h = float(group.select("img_h")[0, 0])

        # Construir las líneas YOLO
        lines = []
        for row in group.iter_rows(named=True):
            cls_id = int(row["class_id"])
            xtl = float(row["bbx_xtl"])
            ytl = float(row["bbx_ytl"])
            xbr = float(row["bbx_xbr"])
            ybr = float(row["bbx_ybr"])

            x_center = ((xtl + xbr) / 2.0) / w
            y_center = ((ytl + ybr) / 2.0) / h
            bw = (xbr - xtl) / w
            bh = (ybr - ytl) / h

            lines.append(f"{cls_id} {x_center:.6f} {y_center:.6f} {bw:.6f} {bh:.6f}")

        # Guardar .txt
        lbl_path = dst_lbl_dir / (Path(img_name).stem + ".txt")
        lbl_path.write_text("\n".join(lines))

    print(f"{dst_img_dir.name} images:", len(list(dst_img_dir.iterdir())))
    print(f"{dst_lbl_dir.name} labels:", len(list(dst_lbl_dir.glob('*.txt'))))


Creamos el directorio de train

In [11]:
IMAGE_SRC_DIR = Path(RAW_DATA_PATH) / "train_images"

ROOT = Path("/kaggle/working/mosquito_yolo")
IMG_TRAIN_DIR = ROOT / "images" / "train"
LBL_TRAIN_DIR = ROOT / "labels" / "train"

build_yolo_split(train_df, IMAGE_SRC_DIR, IMG_TRAIN_DIR, LBL_TRAIN_DIR)

train images: 6396
train labels: 6396


Luego el directorio de val

In [12]:
IMG_VAL_DIR = ROOT / "images" / "val"
LBL_VAL_DIR = ROOT / "labels" / "val"

build_yolo_split(val_df, IMAGE_SRC_DIR, IMG_VAL_DIR, LBL_VAL_DIR)

val images: 800
val labels: 800


Seguido de otro pequeño sanity-check

In [13]:
from pathlib import Path
import os

ROOT = Path("/kaggle/working/mosquito_yolo")

print("ROOT contents:", os.listdir(ROOT))
print("images/train count:", len(list((ROOT / "images" / "train").iterdir())))
print("labels/train count:", len(list((ROOT / "labels" / "train").glob("*.txt"))))
print("images/val   count:", len(list((ROOT / "images" / "val").iterdir())))
print("labels/val   count:", len(list((ROOT / "labels" / "val").glob("*.txt"))))

ROOT contents: ['images', 'labels', 'mosquito.yaml']
images/train count: 6396
labels/train count: 6396
images/val   count: 800
labels/val   count: 800


Por último, debemos construir un mosquitos.yaml para poder entrenar nuestro modelo.

In [14]:
import yaml

data_cfg = {
    "path": str(ROOT),
    "train": "images/train",
    "val": "images/val",
    "names": classes
}

yaml_path = ROOT / "mosquito.yaml"
with open(yaml_path, "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)

print(yaml_path.read_text())

path: /kaggle/working/mosquito_yolo
train: images/train
val: images/val
names:
- aegypti
- albopictus
- anopheles
- culex
- culiseta
- japonicus/koreicus



Podemos ver que ya tenemos todo cargado, por lo que podemos empezar a entregar el modelo

## Entrenamiento

In [15]:
import cv2
from ultralytics.utils import patches

def simple_imread(path, flags=cv2.IMREAD_COLOR):
    return cv2.imread(str(path), flags)

patches.imread = simple_imread
print("Patched ultralytics imread -> cv2.imread")

Patched ultralytics imread -> cv2.imread


In [16]:
from ultralytics import YOLO
import torch

IMG_SIZE = 640
model = YOLO("yolov8s.pt")   # small, fast, still decent

results = model.train(
    data=str(yaml_path),
    imgsz=IMG_SIZE,
    epochs=20,
    batch=16,
    workers=0,
    device=0 if torch.cuda.is_available() else "cpu",
    amp=True,
    patience=5,
    verbose=True,
)

Ultralytics 8.3.228 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/mosquito_yolo/mosquito.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=5, perspective=0.0, plots=T

/usr/local/lib/python3.11/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


train: Scanning /kaggle/working/mosquito_yolo/labels/train... 6395 images, 0 backgrounds, 1 corrupt: 100% ━━━━━━━━━━━━ 6396/6396 1.5Kit/s 4.2s0.1s
train: /kaggle/working/mosquito_yolo/images/train/1294cd51-0b79-4ae2-bda5-58981ee420b2.jpeg: ignoring corrupt image/label: image file is truncated (61 bytes not processed)
train: New cache created: /kaggle/working/mosquito_yolo/labels/train.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2244.2±990.9 MB/s, size: 561.9 KB)
val: Scanning /kaggle/working/mosquito_yolo/labels/val... 800 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 800/800 1.5Kit/s 0.5s0.0ss
val: New cache created: /kaggle/working/mosquito_yolo/labels/val.cache
Plotting labels to /kaggle/working/runs/detect/train2/labels.jpg... 
optimizer: 'optimizer=auto

## Guardado del Modelo
Ahora, para poder guardar nuestro modelo podemos almacenar los mejores weights encontrados dentro del working directory de Kaggle. De esta manera, podemos descargarlos fácilmente y correr el modelo sin necesidad de volver a entrenarlo.

In [18]:
from pathlib import Path
import shutil

ROOT = Path("/kaggle/working")
train_dir = ROOT / "runs" / "detect" / "train2"
weights_dir = train_dir / "weights"

# 1) Copy weights
best_dst = ROOT / "yolov8s_best.pt"
last_dst = ROOT / "yolov8s_last.pt"

shutil.copy(weights_dir / "best.pt", best_dst)
shutil.copy(weights_dir / "last.pt", last_dst)

print("Saved best model to:", best_dst)
print("Saved last model to:", last_dst)

# 2) Copy key plots & metrics
extra_files = [
    "results.png",
    "results.csv",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "BoxF1_curve.png",
    "BoxPR_curve.png",
    "BoxP_curve.png",
    "BoxR_curve.png",
    "labels.jpg",
]

for name in extra_files:
    src = train_dir / name
    if src.exists():
        dst = ROOT / f"train2_{name}"
        shutil.copy(src, dst)
        print(f"Saved {name} to: {dst}")
    else:
        print(f"{name} not found in {train_dir}, skipping.")

Saved best model to: /kaggle/working/yolov8s_best.pt
Saved last model to: /kaggle/working/yolov8s_last.pt
Saved results.png to: /kaggle/working/train2_results.png
Saved results.csv to: /kaggle/working/train2_results.csv
Saved confusion_matrix.png to: /kaggle/working/train2_confusion_matrix.png
Saved confusion_matrix_normalized.png to: /kaggle/working/train2_confusion_matrix_normalized.png
Saved BoxF1_curve.png to: /kaggle/working/train2_BoxF1_curve.png
Saved BoxPR_curve.png to: /kaggle/working/train2_BoxPR_curve.png
Saved BoxP_curve.png to: /kaggle/working/train2_BoxP_curve.png
Saved BoxR_curve.png to: /kaggle/working/train2_BoxR_curve.png
Saved labels.jpg to: /kaggle/working/train2_labels.jpg
